In [20]:
import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torchvision.models import mobilenet_v3_small
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import os

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = mobilenet_v3_small(pretrained=True)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 29)  # 29 ASL classes
model = model.to(device)

In [22]:
transform = transforms.Compose([
    transforms.Resize(224), 
    transforms.CenterCrop(224),  
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [23]:
data_path = 'data/asl_alphabet_train'
dataset = datasets.ImageFolder(data_path, transform=transform)

In [24]:
import random
from torch.utils.data import Subset

subset_size = 10000 
indices = list(range(len(dataset)))
random.seed(42)
random.shuffle(indices)
subset_indices = indices[:subset_size]

dataset = Subset(dataset, subset_indices)

In [32]:
#splits
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_ds,val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=64, num_workers=4, pin_memory=True)


In [26]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr = 0.0001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size= 5, gamma= 0.8)

In [34]:
from tqdm import tqdm
from torch.cuda.amp import GradScaler, autocast

scaler = GradScaler()

num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    total, correct = 0, 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}")
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        loop.set_postfix(loss=loss.item(), accuracy=100.*correct/total)

    train_loss = running_loss / total
    train_acc = 100. * correct / total
    scheduler.step()

    # Validation
    model.eval()
    val_total, val_correct = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with torch.cuda.amp.autocast():
                outputs = model(images)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    val_acc = 100. * val_correct / val_total

    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")


C:\Users\Jomar\AppData\Local\Temp\ipykernel_16828\814625504.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Epoch 1/5:   0%|          | 0/125 [00:00<?, ?it/s]C:\Users\Jomar\AppData\Local\Temp\ipykernel_16828\814625504.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/5: 100%|██████████| 125/125 [01:22<00:00,  1.52it/s, accuracy=100, loss=0.000624]
C:\Users\Jomar\AppData\Local\Temp\ipykernel_16828\814625504.py:42: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1 | Train Loss: 0.0008 | Train Acc: 100.00% | Val Acc: 100.00%


Epoch 2/5: 100%|██████████| 125/125 [01:16<00:00,  1.64it/s, accuracy=100, loss=0.000368]


Epoch 2 | Train Loss: 0.0004 | Train Acc: 100.00% | Val Acc: 100.00%


Epoch 3/5: 100%|██████████| 125/125 [01:17<00:00,  1.62it/s, accuracy=100, loss=0.000207]


Epoch 3 | Train Loss: 0.0003 | Train Acc: 100.00% | Val Acc: 100.00%


Epoch 4/5: 100%|██████████| 125/125 [01:20<00:00,  1.56it/s, accuracy=100, loss=0.00018] 


Epoch 4 | Train Loss: 0.0002 | Train Acc: 100.00% | Val Acc: 100.00%


Epoch 5/5: 100%|██████████| 125/125 [01:18<00:00,  1.60it/s, accuracy=100, loss=0.000135]


Epoch 5 | Train Loss: 0.0001 | Train Acc: 100.00% | Val Acc: 100.00%


In [ ]:
# Save final model
torch.save(model.state_dict(), "asl_model.pth")